In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import plotly.express as px


project_root = Path().resolve().parent
sys.path.append(str(project_root))

from scripts.rq2_function_lib import set_plot_style
from scripts.rq9_function_lib import display_weather_per_region, display_weather_codes_per_region, levene_test_for_extreme_weather 
from scripts.rq9_function_lib import display_levene_test_results, extract_arrays_for_global_test, calculate_global_levene, run_volatility_panel_regression

set_plot_style()

# What impact do extreme weather conditions have on the fuel prices?

In [2]:
weather_path = Path(r'/Users/sebastian/data-science-projekt/weather_per_leitregion')
region_price_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/regions_avg_prices_per_year')
region_coord_path = Path(r'/Users/sebastian/data-science-projekt/plz_leitregionen.csv')

In [3]:
display_weather_per_region(weather_path, "25", "2020")

Reading file from: /Users/sebastian/data-science-projekt/weather_per_leitregion/weather_region25.csv
df successfully created.
Downsampling df...


Display weather codes

In [4]:
display_weather_codes_per_region(weather_path, "25", "2020")

Reading file from: /Users/sebastian/data-science-projekt/weather_per_leitregion/weather_region25.csv
df successfully created.
Downsampling df...


Now we want to check with a Levene test, if an extreme weather event influences the fuel prices. For that we calculated the median price for each german 'Postleitzahl-Leitregion' and collected the historical weather data for the geographical middle of each region.

Weiter und genauer erklären!!!!

In [3]:
regions_df = pd.read_csv(region_coord_path, dtype={"leit_plz":str}) 
regions = regions_df["leit_plz"].to_list()


In [6]:
all_regional_results = []
global_arrays = {}

for region in regions:
    # do regional test first
    result_list = levene_test_for_extreme_weather(weather_path, region_price_path, region)
    all_regional_results.extend(result_list)

    # collect arrays for gloabl test
    arrays_global_levene = extract_arrays_for_global_test(weather_path, region_price_path, region)

    if arrays_global_levene["status"] == "Success":
        for var_name, arrays in arrays_global_levene["arrays"].items():
            if var_name not in global_arrays:
                global_arrays[var_name] = {"event": [], "control": []}
            
            global_arrays[var_name]["event"].append(arrays["event"])
            global_arrays[var_name]["control"].append(arrays["control"])

print("Data collected. Calculating global levene...")
global_results = calculate_global_levene(global_arrays)

#convert to pandas df
global_result_df = pd.DataFrame(global_results)
result_df = pd.DataFrame(all_regional_results)


Data collected. Calculating global levene...


Regional results of the Levene test.

In [7]:
#display_levene_test_results(result_df)

In [8]:
df_map_data = pd.merge(result_df, regions_df, left_on = "region", right_on = "leit_plz", how="inner")

df_diesel_plot = df_map_data[df_map_data["variable"] == "diesel_median"].copy()
df_e5_plot = df_map_data[df_map_data["variable"] == "e5_median"].copy()
df_e10_plot = df_map_data[df_map_data["variable"] == "e10_median"].copy()

In [9]:
fig = px.scatter_map(
    df_diesel_plot,
    lat = "avg_lat",
    lon = "avg_lng",
    color = "significant", 
    color_discrete_map={True: "mediumseagreen", False: "lightcoral"},
    
    hover_name = "region",
    hover_data={
        "avg_lat": False,
        "avg_lng": False,
        "p_value": "{:.4f}",
        "test_statistic": "{:.2f}",
        "significant": True
    },
    zoom = 4,
    map_style = "open-street-map",
    title = "Regional Significance: influence of extreme weather on diesel prices"
)

fig.update_layout(legend_title_text = "significant effect (p < 0.05)")

fig.show()

In [10]:
fig = px.scatter_map(
    df_e5_plot,
    lat = "avg_lat",
    lon = "avg_lng",
    color = "significant", 
    color_discrete_map={True: "mediumseagreen", False: "lightcoral"},
    
    hover_name = "region",
    hover_data={
        "avg_lat": False,
        "avg_lng": False,
        "p_value": "{:.4f}",
        "test_statistic": "{:.2f}",
        "significant": True
    },
    zoom = 4,
    map_style = "open-street-map",
    title = "Regional Significance: influence of extreme weather on e5 prices"
)

fig.update_layout(legend_title_text = "significant effect (p < 0.05)")

#fig.show()

In [11]:
fig = px.scatter_map(
    df_e10_plot,
    lat = "avg_lat",
    lon = "avg_lng",
    color = "significant", 
    color_discrete_map={True: "mediumseagreen", False: "lightcoral"},
    
    hover_name = "region",
    hover_data={
        "avg_lat": False,
        "avg_lng": False,
        "p_value": "{:.4f}",
        "test_statistic": "{:.2f}",
        "significant": True
    },
    zoom = 4,
    map_style = "open-street-map",
    title = "Regional Significance: influence of extreme weather on e10 prices"
)

fig.update_layout(legend_title_text = "significant effect (p < 0.05)")

#fig.show()

Result for the Levene test calculated over all of Germany

TODO: texte ordentlich & Methoden Kommentare

In [12]:
display_levene_test_results(global_result_df)

,variable,test_statistic,p_value,significant,data_event,data_controll,note
0,diesel_median,7.03,0.0080,True,4156749,1093198,Global Success!
1,e5_median,8.51,0.0035,True,4156749,1093198,Global Success!
2,e10_median,13.43,0.0002,True,4156749,1093198,Global Success!


panel regression test for diesel to see if we continue pursuing this rq

In [4]:
mein_modell=run_volatility_panel_regression(region_price_path,weather_path,regions,)

1. Lade und verarbeite regionale Daten für das Panel...
2. Setze Panel-Datensatz zusammen...
Panel erstellt! 8,684,102 Zeilen. Konvertiere für pyfixest...
3. Berechne Fixed Effects Regression & Clustered Standard Errors...


/Users/sebastian/Desktop/CAU Studium/3semester/DataScienceProject/Data-Science-Projekt/venv/lib/python3.12/site-packages/pyfixest/estimation/model_matrix_fixest_.py:181: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  df[cols_to_convert] = df[cols_to_convert].astype("float64")



 ERGEBNIS DER PANEL-REGRESSION
###

Estimation:  OLS
Dep. var.: volatilitaet, Fixed effects: region+datum+stunde
Inference:  CRV1
Observations:  8684102

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| extremwetter  |      0.000 |        0.000 |     0.212 |      0.833 | -0.001 |   0.001 |
---
RMSE: 0.131 R2: 0.106 R2 Within: 0.0 
